## Context

This notebook accompanies User Guide section 5 (GMS Diagnostics).  It runs `GMSDiagnostics` on the prebuilt fixture and visualizes the per-check pass/fail grid.  A failing diagnostic almost never points at the knowledge document — it usually points at a training config or a numerical-stability regression in the GMS core.  That distinction is what makes the diagnostic report worth running on every store you ship.

In [ ]:
# Phase 1B preamble: bootstrap runtime + load shared helpers
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# tutorials/user_guide/notebooks/_shared.py. Works in both the source
# clone (knowlytix/GMS) and the public mirror clone (knowlytix/gmsh).
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'tutorials' / 'user_guide' / 'notebooks' / '_shared.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from tutorials.user_guide.notebooks import _helpers_phase1 as h1
h1.bootstrap_runtime()  # stubs modeva if absent
from tutorials.user_guide.notebooks import _shared as ug
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: post-training invariants the store must satisfy

`GMSDiagnostics.run_all` returns a `DiagnosticReport` with one `CheckResult` per invariant.  The checks fall into two groups.  **Structural invariants** verify that training did not break the geometry: relation operators stay orthogonal, projection matrices stay on the Stiefel manifold, and entity embeddings stay on the unit sphere.  **Primitive invariants** verify each of the six GMS primitives produces meaningful output: `score_triple` discriminates known from fabricated, `tension_energy` separates agreement from contradiction across the $\sqrt{2}$ boundary, `check_holonomy` stays well below the random baseline of $\sqrt{2m}$, ENM passes byte-identical SHA-256 round-trip, `link_predict` ranks the true tail near the top, and `query_triples` is exact against the document graph.

When a check fails, fix the upstream cause before trusting any other primitive.  An orthogonality violation, for instance, breaks every downstream geodesic; reading `score_triple` after that is meaningless.  Most regressions surface here first.

In [ ]:
from knowlytix.core.diagnostics import GMSDiagnostics

store = ug.load_ds_practices_store()
diag = GMSDiagnostics(
    model=store.model,
    adapter=store.adapter,
    enm=store.enm,
    doc_graph=store.doc_graph,
    device=store.device,
)
report = diag.run_all()
print(f'Total checks: {report.total}, '
      f'passed: {report.pass_count}, '
      f'all_passed: {report.all_passed}')


In [ ]:
GMSDiagnostics.print_report(report)


In [ ]:
# Visualization: pass/fail grid, one row per check
import matplotlib.pyplot as plt

names = [c.name for c in report.checks]
passed = [bool(c.passed) for c in report.checks]
fig, ax = plt.subplots(figsize=(7, 0.5 * len(names) + 1.5))
h1.plot_pass_fail_grid(
    ax, names, passed,
    title=f'GMSDiagnostics: {report.pass_count}'
          f'/{report.total} passed',
)
fig.tight_layout()
plt.show()


### Exercises

1. Find the lowest-`score` check in `report.checks`.  What primitive does it cover, and what would you investigate first if it slipped below the pass threshold?
2. Reduce `sample_size` to 50 in the `GMSDiagnostics` constructor and rerun.  Do any checks change verdict?  If yes, the check is sample-noise sensitive.
3. Print `report.checks[3].evidence` (the `score_triple` check) and identify three failures that would be useful in a debugging report.

### What you should now be able to do

- Run `GMSDiagnostics` on a built store.
- Read `report.pass_count`, `report.total`, and   `report.all_passed`.
- Read each `CheckResult.detail` and `evidence` to   diagnose a specific failure.
- Recognise that a failing check usually indicates an   upstream training or geometry issue, not a knowledge   data issue.

### Cross-references

- USER_GUIDE.md section 5 (GMS Diagnostics)
- `gms/diagnostics.py:GMSDiagnostics.run_all`
- `gms_agent_builder_ext/chapters/ch04.tex` for the   full discussion of post-training invariants.

In [ ]:
# Final assertion: structural invariants and the
# byte-deterministic primitives (lookup_enm,
# query_triples, check_holonomy) pass on the regex-mode
# fixture.  score_triple, tension_energy, and
# link_predict are corpus-density-sensitive and reach
# their pass thresholds only on hybrid-mode fixtures;
# tighten this assertion to `pass_count == total` once
# the fixture is rebuilt with ANTHROPIC_API_KEY set.
must_pass = {
    'operator_orthogonality',
    'projection_integrity',
    'embedding_norm',
    'check_holonomy',
    'lookup_enm',
    'query_triples',
}
by_name = {c.name: c for c in report.checks}
missing = must_pass - set(by_name)
assert not missing, f'expected checks not run: {missing}'
failed = [n for n in must_pass if not by_name[n].passed]
assert not failed, (
    f'core diagnostics failed: {failed}; full report: '
    + str([(c.name, c.passed) for c in report.checks])
)
print(
    f'OK: diagnostics {report.pass_count}/{report.total} '
    f'overall; all {len(must_pass)} core invariants pass.'
)
